# 02 — PCP: collect → train Q-corrector → 3-way eval

Same orchestration pattern as notebook 01: the loops are visible here; the package
provides `run_episode` (driven by `RolloutConfig(pcp=...)` flags) + `store` + training.

## 1. Secrets + install

In [ ]:
import os
from google.colab import userdata
for k in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN', 'WANDB_API_KEY'):
    os.environ[k] = userdata.get(k)
GH_PAT = userdata.get('GH_PAT')
![ -d pnp-vla ] || git clone -q https://$GH_PAT@github.com/ArjunS07/pnp-vla.git
!cd pnp-vla && git pull -q && pip install -q -e '.[sim]'

## 2. Environment + model + store + PRO episodes

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()

In [ ]:
from pnp import libero_env, libero_pro, models, PCPConfig, RolloutConfig
from pnp.store import SupabaseStore
from pnp.rollout import run_episode, iter_task_envs

libero_env.init_libero_benchmark()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()
EXPERIMENT = 'pcp-v1'
pcp = PCPConfig()

# libero_pro.apply_env_patches(); libero_pro.patch_torch_load()
# bd = libero_pro.reload_benchmark()
# pro_eps = libero_pro.build_libero_pro_episodes(bd, episode_idxs=range(10))
pro_eps = []   # <- fill from libero_pro once assets are set up

## 3. Collect labeled (z_hat, obs_enc) chunks → qc_rollouts

In [ ]:
COLLECT = RolloutConfig(pcp=PCPConfig(mode='collect', collect_steps=pcp.collect_steps,
                                      pnp_k=pcp.pnp_k), record_trajectory=False)
store.start_run(driver='pcp_collect', benchmark='libero_pro', experiment=EXPERIMENT)
for env, task_eps in iter_task_envs(pro_eps):
    for ep in task_eps:
        rid = f"{ep['suite']}:{ep['task_idx']}:{ep['ep_idx']}:{ep['init_state_hash']}"
        res = run_episode(env, ep, policy, preprocess, device, COLLECT)
        store.log_collect(rid, ep, res)
store.finish_run()

## 4. Train + calibrate the Q-corrector (wandb) → q_correctors

In [ ]:
import wandb
from pnp.pcp import load_qc_samples, train_q_corrector, ckpt_bytes, new_q_ckpt_id

qc_rows = store.load_qc_rows(experiment=EXPERIMENT)
samples = load_qc_samples(qc_rows, pcp)
run = wandb.init(project='pnp-qcorrector', name=EXPERIMENT, config={'experiment': EXPERIMENT})
q_model, q_scaler, meta, split_ids = train_q_corrector(
    samples, device, cfg=pcp, wandb_run=run, experiment=EXPERIMENT)
q_ckpt_id = new_q_ckpt_id()
store.register_q_corrector(q_ckpt_id, ckpt_bytes(q_model, q_scaler, meta), meta, split_ids=split_ids)
run.finish()
print('q_ckpt_id =', q_ckpt_id, ' val_auc =', meta['val_auc'])

## 5. 3-way eval (vanilla / pnp-only / pcp) → qc_eval

The three arms are just `RolloutConfig`s — vanilla, `mode='correct'` with λ=0 (P&P refine,
no gradient), and λ>0 (full PCP). The corrector is attached to the config's runtime handles.

In [ ]:
from pnp.pcp import QCorrector, TemperatureScaler

ckpt, _ = store.load_q_corrector(q_ckpt_id)
q_model = QCorrector(ckpt['action_dim'], ckpt['obs_dim']).to(device)
q_model.load_state_dict(ckpt['model']); q_model.eval()
q_scaler = TemperatureScaler().to(device); q_scaler.load_state_dict(ckpt['scaler'])

def correct_cfg(lam):
    return RolloutConfig(pcp=PCPConfig(mode='correct', lambda_pcp=lam, correction_steps=pcp.correction_steps,
                                       pnp_k=pcp.pnp_k, q_gate=pcp.q_gate, q_model=q_model,
                                       q_scaler=q_scaler, q_ckpt_id=q_ckpt_id))
PASSES = [('vanilla', -1.0, RolloutConfig()),
          ('pnp_only', 0.0, correct_cfg(0.0)),
          ('pcp', pcp.lambda_pcp, correct_cfg(pcp.lambda_pcp))]

store.start_run(driver='pcp_eval', benchmark='libero_pro', experiment=EXPERIMENT)
for env, task_eps in iter_task_envs(pro_eps):
    for name, lam, cfg in PASSES:
        for ep in task_eps:
            rid = f"{ep['suite']}:{ep['task_idx']}:{ep['ep_idx']}:{ep['init_state_hash']}"
            res = run_episode(env, ep, policy, preprocess, device, cfg)
            store.log_eval(rid, ep, lam, res)
store.finish_run()